# 多头注意力机制

给定相同的 Query、Key、Value，希望模型能够学习不同的行为，比如有的注意力头关注短距离依赖，有的关注长距离依赖；因此把 Query、Key、Value 投影到不同的表示子空间，然后分别做注意力。

$$ \mathbf{h}_i = f\left(\mathbf{W}_i^{(q)} \mathbf{q}, \mathbf{W}_i^{(k)} \mathbf{k}, \mathbf{W}_i^{(v)} \mathbf{v}\right) \in \mathbb{R}^{p_v} $$

* $\mathbf{q} \in \mathbb{R}^{d_q}$：输入的查询向量（Query）；
* $\mathbf{k} \in \mathbb{R}^{d_k}$：输入的键向量（Key）；
* $\mathbf{v} \in \mathbb{R}^{d_v}$：输入的值向量（Value）  

f函数相当于去做普通的缩放点积注意力机制：
$$
head_i = Attention(Q_i, K_i, V_i) = softmax(\frac{Q_i^T K_i^T} {\sqrt{d_h}}) \cdot V_i

计算完多头注意力之后，把所有头的输出拼接起来，再经过一个线性层$W_o$，得到最终的输出

$$MultiHeadAttention = W_o [head1, head2, ..., head_h]$$

$$\mathbf{W}_o \begin{bmatrix} \mathbf{h}_1 \\ \vdots \\ \mathbf{h}_h \end{bmatrix} \in \mathbb{R}^{p_o} $$

## 代码实现

先把QKV投影到num_hiddens层，把最后一维拆成多个头， 然后把头并到batch维度上，一次性调用缩放点积注意力

* transpose_qkv: 拆解多头维度， 把每个头当做一个独立的batch，一次性送入后面的点积注意力机制
* transpose_output: 拼接多头维度， 把每个头的输出拼接起来， 重新恢复原始的形状

In [2]:
import math
import torch
from torch import nn
from d2l import torch as d2l

def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算而变换形状"""
    # 输入X的形状:(batch_size，查询或者“键－值”对的个数，num_hiddens)
    # 输出X的形状:(batch_size，查询或者“键－值”对的个数，num_heads，
    # num_hiddens/num_heads)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # 输出X的形状:(batch_size，num_heads，查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    X = X.permute(0, 2, 1, 3)

    # 最终输出的形状:(batch_size*num_heads,查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])

#@save
def transpose_output(X, num_heads):  # X.shape = (batch_size*num_heads, query_num, num_hiddens/num_heads)
    """逆转transpose_qkv函数的操作"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)  # X.shape = (batch_size, query_num, num_hiddens)

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


实现多头注意力机制

In [3]:
#@save
class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = d2l.DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # queries，keys，values的形状:
        # (batch_size，查询或者“键－值”对的个数，num_hiddens)
        # valid_lens　的形状:
        # (batch_size，)或(batch_size，查询的个数)
        # 经过变换后，输出的queries，keys，values　的形状:
        # (batch_size*num_heads，查询或者“键－值”对的个数，
        # num_hiddens/num_heads)
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)

        if valid_lens is not None:
            # 在轴0，将第一项（标量或者矢量）复制num_heads次，
            # 然后如此复制第二项，然后诸如此类。
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)

        # output的形状:(batch_size*num_heads，查询的个数，
        # num_hiddens/num_heads)
        output = self.attention(queries, keys, values, valid_lens)

        # output_concat的形状:(batch_size，查询的个数，num_hiddens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)